# AI Cycling Coach — GPU Training (Colab)

**Runtime → Change runtime type → T4 GPU** before running.

## Just click "Run all" — no uploads needed

The notebook generates synthetic training data directly here in Colab (~3 min for 20K athletes on CPU), then immediately trains on the GPU.

Steps:
1. Check GPU
2. Clone repo + install deps
3. Generate 20K athletes (~3 min on Colab CPU) — or change `ATHLETES = 50_000` for a fuller dataset (~8 min)
4. Train on GPU (~1–2 h on T4)
5. Save model to Drive + push to GitHub

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO = 'https://github.com/yossibello/ai-coach.git'

if not os.path.exists('/content/ai-coach'):
    !git clone {REPO} /content/ai-coach
else:
    !cd /content/ai-coach && git pull

%cd /content/ai-coach

import sys
sys.path.insert(0, '/content/ai-coach/backend')
os.environ['PYTHONPATH'] = '/content/ai-coach/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
!pip install pandas pyarrow -q
# torch is pre-installed on Colab with CUDA support
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ── 4. Load / generate training data ─────────────────────────────────────────
# THREE OPTIONS — set the MODE variable below:
#
#   'generate'  → generate data here in Colab (no upload needed, ~3 min)
#   'upload'    → you manually uploaded synthetic.parquet via the sidebar
#   'drive'     → file is on Google Drive at DRIVE_PATH

import os, shutil, pandas as pd

MODE       = 'generate'   # ← 'generate' | 'upload' | 'drive'
ATHLETES   = 20_000       # only used when MODE='generate'  (20K ≈ 3 min, 50K ≈ 8 min)
DRIVE_PATH = '/content/drive/MyDrive/ai-coach-data/synthetic.parquet'
DATA_FILE  = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if os.path.exists(DATA_FILE):
    print('✓ Data already in place, skipping.')

elif MODE == 'generate':
    import multiprocessing
    workers = max(1, multiprocessing.cpu_count() - 1)
    print(f'Generating {ATHLETES:,} athletes using {workers} workers…')
    !python -m ml.training.generate_synthetic \
        --athletes {ATHLETES} \
        --workers  {workers} \
        --output   {DATA_FILE}

elif MODE == 'upload':
    UPLOAD_PATH = '/content/synthetic.parquet'
    if not os.path.exists(UPLOAD_PATH):
        raise FileNotFoundError('Upload synthetic.parquet via sidebar → Files → Upload first')
    print(f'Copying uploaded file ({os.path.getsize(UPLOAD_PATH)/1e6:.0f} MB)…')
    shutil.copy(UPLOAD_PATH, DATA_FILE)

elif MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Copying from Drive ({os.path.getsize(DRIVE_PATH)/1e6:.0f} MB)…')
    shutil.copy(DRIVE_PATH, DATA_FILE)

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'pc_5s_wkg'       in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
print(f'✓ Data ready: {len(df):,} rows, {df.athlete_id.nunique():,} athletes, {len(df.columns)} cols')
del df


In [ ]:

# ── 5. Train ──────────────────────────────────────────────────────────────────
import sys, os, torch

# ── Step 1: GPU info ──────────────────────────────────────────────────────────
print("=== Step 1: GPU ===")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if torch.cuda.is_available():
    print(f"  CUDA: True  |  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.1f} GB")
else:
    print("  CUDA: False — training will be VERY slow on CPU")
sys.stdout.flush()

if   vram_gb >= 45: BATCH_SIZE = 4096  # A6000 48 GB / A100 80 GB
elif vram_gb >= 38: BATCH_SIZE = 2048  # A100 40 GB / H100
elif vram_gb >= 20: BATCH_SIZE = 1024  # RTX 3090+
else:               BATCH_SIZE = 1024  # T4 16 GB — 8M-param model uses ~3 GB

STEPS_PER_EPOCH = 3000   # 3.07M samples/epoch (6× vs old 2000×256 setup)
EPOCHS          = 50     # each epoch sees 6× more data
MODEL_FILE      = 'backend/models/cycling_coach.pt'
print(f"  batch_size={BATCH_SIZE}  steps_per_epoch={STEPS_PER_EPOCH}  epochs={EPOCHS}")
sys.stdout.flush()

# ── Step 2: Working directory ─────────────────────────────────────────────────
print("\n=== Step 2: Working directory ===")
print(f"  cwd: {os.getcwd()}")
if os.getcwd() != '/content/ai-coach':
    os.chdir('/content/ai-coach')
    print(f"  → changed to: {os.getcwd()}")
sys.stdout.flush()

# ── Step 3: sys.path ──────────────────────────────────────────────────────────
print("\n=== Step 3: sys.path ===")
for p in ['/content/ai-coach/backend', '/content/ai-coach']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('\n'.join(f"  {p}" for p in sys.path[:8]))
sys.stdout.flush()

# ── Step 4: Data file ─────────────────────────────────────────────────────────
print("\n=== Step 4: Data file ===")
DATA_FILE = locals().get('DATA_FILE', 'ml/data/synthetic.parquet')
print(f"  DATA_FILE = {DATA_FILE!r}")
abs_data  = os.path.abspath(DATA_FILE)
print(f"  abs path  = {abs_data!r}")
print(f"  exists    = {os.path.exists(abs_data)}")
if os.path.exists(abs_data):
    print(f"  size      = {os.path.getsize(abs_data)/1e6:.0f} MB")
else:
    ml_data = os.path.join(os.getcwd(), 'ml', 'data')
    if os.path.isdir(ml_data):
        print(f"  ml/data/ contents: {os.listdir(ml_data)}")
    raise FileNotFoundError(
        f"Data file not found: {abs_data}\nRe-run cell 4 (data/generate cell) first!"
    )
sys.stdout.flush()

# ── Step 5: Validate parquet columns ─────────────────────────────────────────
print("\n=== Step 5: Parquet check ===")
import pandas as pd
df_check = pd.read_parquet(abs_data, columns=['athlete_id', 'risk_ot_class', 'risk_inj_target', 'pc_5s_wkg'])
print(f"  rows: {len(df_check):,}  athletes: {df_check.athlete_id.nunique():,}")
print(f"  required columns: OK")
del df_check
sys.stdout.flush()

# ── Step 6: Import training module ────────────────────────────────────────────
print("\n=== Step 6: Importing ml.training.train ===")
try:
    from ml.training.train import train as run_training
    print("  Import: OK")
except ImportError as e:
    print(f"  Import FAILED: {e}")
    raise
sys.stdout.flush()

# ── Step 7: Run training ──────────────────────────────────────────────────────
print("\n=== Step 7: Training ===")
os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

import argparse
args = argparse.Namespace(
    data             = DATA_FILE,
    output           = MODEL_FILE,
    checkpoint       = None,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    steps_per_epoch  = STEPS_PER_EPOCH,
    lr               = 3e-4,
    seq_len          = 90,
    val_frac         = 0.1,
    seed             = 42,
    d_model          = 256,   # was 128 — 2× wider
    nhead            = 8,
    num_layers       = 8,     # was 6
    d_ff             = 1024,  # was 512
    dropout          = 0.1,
    fast             = False,
    patience         = 20,
    compile          = False,
    no_amp           = False,
)
print("  args OK — starting training loop…")
sys.stdout.flush()

run_training(args)
print(f"\n✓ Training complete! Model → {MODEL_FILE}")


In [ ]:
# ── 6a. Save model to Google Drive (survives session end) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/ai-coach-models/'
os.makedirs(dst, exist_ok=True)
shutil.copy('backend/models/cycling_coach.pt', dst)
print('Saved to Google Drive:', dst + 'cycling_coach.pt')

In [ ]:
# ── 6b. Push model back to GitHub ─────────────────────────────────────────────
# You need a GitHub Personal Access Token (PAT) with repo write access.
# Create one at: https://github.com/settings/tokens  (Classic, repo scope)

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'colab@training'
!git config user.name 'Colab Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Colab GPU)"
!git push origin main
print('Model pushed to GitHub!')

In [ ]:
# ── 7. Quick sanity check ─────────────────────────────────────────────────────
import torch, sys
sys.path.insert(0, '/content/ai-coach/backend')
from app.ml.model import CyclingTransformer

ckpt = torch.load('backend/models/cycling_coach.pt', map_location='cpu')
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model         = cfg.get('d_model', 256),
    nhead           = cfg.get('nhead', 8),
    num_layers      = cfg.get('num_layers', 8),
    dim_feedforward = cfg.get('dim_feedforward', 1024),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()
metrics = ckpt.get('metrics', {})
print('Model loaded OK')
print(f'Parameters:  {sum(p.numel() for p in m.parameters()):,}')
print(f'Best epoch:  {metrics.get("epoch",    "n/a")}')
print(f'Val loss:    {metrics.get("val_loss", "n/a")}')
print(f'wt_acc:      {metrics.get("wt_acc",   "n/a")} %')
print(f'FTPΔ MAE:    {metrics.get("ftp_mae",  "n/a")} W')
